### Build Races Dimension (`dim_races`)

This notebook builds the **Races Dimension Table** in the gold layer of our Formula 1 data lakehouse. It is part of the **medallion architecture** pipeline (bronze -> silver -> gold) and sits in the `04-gold` folder.

##### What this notebook does:
1. **Reads** two cleaned tables from the silver layer - `circuits` and `races`
2. **Joins** them on `circuit_id` to combine race event details with circuit location information
3. **Selects** only the columns needed for the dimension: season, round, race_name, race_date, circuit_name, locality, and country
4. **Writes** the result as a Delta table to `formula1.gold.dim_races`

##### Purpose:
The `dim_races` table serves as a denormalized dimension table that allows analysts and dashboards to query race and circuit information in a single table - without needing to perform joins at query time.

##### Dependencies:
- **Upstream config:** **`00-common/01.environment-config`** (provides `catalog_name`, `silver_schema`, `gold_schema` variables)
- **Source tables:** `formula1.silver.circuits`, `formula1.silver.races`
- **Target table:** `formula1.gold.dim_races`

In [0]:
%run ../00-common/01.environment-config 

##### Step 0: Import Required Functions
Here we import all the built-in functions from `pyspark.sql.functions` (such as `col`, `lit`, `when`, `concat`, etc.). This gives us access to all the transformation and aggregation functions we might need when working with our DataFrames later in the notebook.

In [0]:
from pyspark.sql.functions import *

##### Define the Gold Target Table
We define the fully qualified target table name where our final dimension table will be written. The variable `target_table` is set to `formula1.gold.dim_races` by combining:
- `catalog_name` -> the Unity Catalog name (`formula1`)
- `gold_schema` -> the gold layer schema (`gold`)
- Table name -> `dim_races`

This follows the **medallion architecture** (bronze -> silver -> gold), where the gold layer holds clean, business-ready dimension and fact tables.

In [0]:
#silver_table = f{catalog_name}.{sil}
target_table = f'{catalog_name}.{gold_schema}.dim_races'

##### Step 1: Read the Silver Layer Tables
We read two tables from the **silver schema** into Spark DataFrames:
- `circuits_df` -> reads `formula1.silver.circuits` which contains circuit details (circuit name, location, country)
- `races_df` -> reads `formula1.silver.races` which contains race details (season, round, race name, race date, circuit_id)

Both DataFrames are loaded lazily (Spark won't actually read the data until an action like `display()` or `write` is triggered). The `circuit_id` column exists in both tables and will be used as the join key in the next step.

In [0]:
circuits_df = spark.table(f'{catalog_name}.{silver_schema}.circuits')
races_df = spark.read.table(f'{catalog_name}.{silver_schema}.races')

##### Step 2: Join Races with Circuits
We create a new DataFrame called `dim_races_df` by performing an **inner join** between `races_df` and `circuits_df` on the `circuit_id` column. This combines race information with circuit location details.

**Join condition:** `races_df.circuit_id == circuits_df.circuit_id`

**Selected columns from the joined result:**
| Column | Source DataFrame | Description |
|--------|-----------------|-------------|
| `season` | races_df | The year/season of the race |
| `round` | races_df | The round number within the season |
| `race_name` | races_df | Name of the race (e.g., "British Grand Prix") |
| `race_date` | races_df | Date the race took place |
| `circuit_name` | circuits_df | Name of the circuit (e.g., "Silverstone") |
| `locality` | circuits_df | City/town where the circuit is located |
| `country` | circuits_df | Country where the circuit is located |

The inner join ensures only races with a matching circuit are included in the result.

In [0]:
dim_races_df = (
    races_df
    .join(circuits_df,
           races_df.circuit_id == circuits_df.circuit_id,
           'inner'
          )
    .select(races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.race_date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

##### Step 3: Write to Gold Layer
Finally, we write the `dim_races_df` DataFrame to the gold schema as a Delta table called `dim_races`. This table serves as a **dimension table** that combines race and circuit information into a single, denormalized table ready for analytics and reporting.

- **Format:** Delta (supports ACID transactions, time travel, and schema evolution)
- **Mode:** Overwrite (replaces the entire table with fresh data each run)
- **Target:** `formula1.gold.dim_races`

In [0]:
(
    dim_races_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))